In [1]:

# 2024-06-12: .env 파일에서 키움 API 설정값 불러오기 (테스트/예제용)
import os
from pathlib import Path
from dotenv import load_dotenv
from pykiwoom_rest import KiwoomRest
# Jupyter 환경에서는 __file__ 미정의 → 현재 워킹 디렉토리 기준으로 상위 폴더의 .env 지정
env_path = (Path.cwd().parent / ".env").resolve()
load_dotenv(dotenv_path=env_path)

ACCOUNT_NO = os.getenv("ACCOUNT_NO")
KIWOOM_APPKEY = os.getenv("KIWOOM_APPKEY")
KIWOOM_APPSECRET = os.getenv("KIWOOM_APPSECRET")


KiwoomAPI = KiwoomRest(
    account_no=ACCOUNT_NO,
    appkey=KIWOOM_APPKEY,
    appsecret=KIWOOM_APPSECRET,
    env_path=env_path,
    use_mock=False,
)

if KiwoomAPI:
    print("API initialized successfully")


API initialized successfully


In [2]:
frgn = KiwoomAPI.get_foreign_trading("005930")

def parse_float(val):
    if isinstance(val, str):
        val = val.replace("+", "").replace("-", "").replace("%", "")
        try:
            return float(val)
        except Exception:
            return 0.0
    return 0.0

def parse_int(val):
    if isinstance(val, str):
        val = val.replace("+", "").replace("-", "").replace(",", "")
        try:
            return int(val)
        except Exception:
            return 0
    return 0

frgn_list = frgn.get('stk_frgnr', [])

# 날짜 역순(최신→과거)이므로 분석을 위해 순서를 뒤집음
frgn_list_rev = list(reversed(frgn_list))

wghts = [parse_float(item.get('wght', '0')) for item in frgn_list_rev]
trde_qtys = [parse_int(item.get('trde_qty', '0')) for item in frgn_list_rev]
dates = [item.get('dt', None) for item in frgn_list_rev]

def get_period_trend(seq, period):
    if len(seq) < period + 1:
        return None, None, "데이터 부족"
    start = seq[-(period+1)]
    end = seq[-1]
    diff = end - start
    if diff > 0:
        trend = "증가"
    elif diff < 0:
        trend = "감소"
    else:
        trend = "유지"
    return start, end, trend

def get_period_dates(period):
    if len(dates) < period + 1:
        return None, None
    return dates[-(period+1)], dates[-1]

periods = [30, 14, 7, 3]
result = {}

for p in periods:
    w_start, w_end, w_trend = get_period_trend(wghts, p)
    q_start, q_end, q_trend = get_period_trend(trde_qtys, p)
    d_start, d_end = get_period_dates(p)
    result[f"{p}일_비중_시작"] = w_start
    result[f"{p}일_비중_종료"] = w_end
    result[f"{p}일_비중_추세"] = w_trend
    result[f"{p}일_거래량_시작"] = q_start
    result[f"{p}일_거래량_종료"] = q_end
    result[f"{p}일_거래량_추세"] = q_trend
    result[f"{p}일_시작일"] = d_start
    result[f"{p}일_종료일"] = d_end

result["최근_비중_시퀀스"] = wghts[-5:]
result["최근_거래량_시퀀스"] = trde_qtys[-5:]

result

{'30일_비중_시작': 50.54,
 '30일_비중_종료': 52.26,
 '30일_비중_추세': '증가',
 '30일_거래량_시작': 21928565,
 '30일_거래량_종료': 20899788,
 '30일_거래량_추세': '감소',
 '30일_시작일': '20250910',
 '30일_종료일': '20251029',
 '14일_비중_시작': 51.85,
 '14일_비중_종료': 52.26,
 '14일_비중_추세': '증가',
 '14일_거래량_시작': 49883028,
 '14일_거래량_종료': 20899788,
 '14일_거래량_추세': '감소',
 '14일_시작일': '20251002',
 '14일_종료일': '20251029',
 '7일_비중_시작': 52.13,
 '7일_비중_종료': 52.26,
 '7일_비중_추세': '증가',
 '7일_거래량_시작': 17589756,
 '7일_거래량_종료': 20899788,
 '7일_거래량_추세': '증가',
 '7일_시작일': '20251020',
 '7일_종료일': '20251029',
 '3일_비중_시작': 52.22,
 '3일_비중_종료': 52.26,
 '3일_비중_추세': '증가',
 '3일_거래량_시작': 18801925,
 '3일_거래량_종료': 20899788,
 '3일_거래량_추세': '증가',
 '3일_시작일': '20251024',
 '3일_종료일': '20251029',
 '최근_비중_시퀀스': [52.13, 52.22, 52.36, 52.3, 52.26],
 '최근_거래량_시퀀스': [18488581, 18801925, 22169970, 20002282, 20899788]}

In [3]:
program = KiwoomAPI.get_hourly_program_trading_paginated("005930", "20250922", "1")['output']

# 순매수금액(prm_netprps_amt), 순매수금액증감(prm_netprps_amt_irds) 기반 추세 분석
def parse_amount(val):
    # +, - 기호 및 쉼표 제거 후 int 변환
    if isinstance(val, str):
        val = val.replace("+", "").replace(",", "")
        try:
            return int(val)
        except Exception:
            return 0
    return 0

# 순매수금액 리스트
net_amounts = [parse_amount(tick.get('prm_netprps_amt', '0')) for tick in program]
# 순매수금액증감 리스트
net_amount_changes = [parse_amount(tick.get('prm_netprps_amt_irds', '0')) for tick in program]

# 추세 판단: 단순히 첫 값과 마지막 값 비교 (증가/감소/유지)
if not net_amounts or len(net_amounts) < 2:
    trend = "데이터 부족"
else:
    if net_amounts[-1] > net_amounts[0]:
        trend = "순매수금액 증가"
    elif net_amounts[-1] < net_amounts[0]:
        trend = "순매수금액 감소"
    else:
        trend = "순매수금액 변화 없음"

# 결과 출력
{
    "시작시각": program[0]['tm'] if program else None,
    "종료시각": program[-1]['tm'] if program else None,
    "시작_순매수금액": net_amounts[0] if net_amounts else None,
    "종료_순매수금액": net_amounts[-1] if net_amounts else None,
    "총_증감": net_amounts[-1] - net_amounts[0] if len(net_amounts) >= 2 else None,
    "추세": trend,
    "순매수금액증감_시퀀스": net_amount_changes[:5],  # 앞 5개만 예시로
}

{'시작시각': '153023',
 '종료시각': '085003',
 '시작_순매수금액': 0,
 '종료_순매수금액': 0,
 '총_증감': 0,
 '추세': '순매수금액 변화 없음',
 '순매수금액증감_시퀀스': [0, 0, 0, 0, 0]}

## 프로그램매매 추세 분석 (고도화 버전)

장 시작부터 현재까지의 프로그램매매 데이터를 수집하여 스마트하게 추세를 분석합니다.
- 전반부/후반부 평균 비교
- 선형 회귀 기울기 분석
- 변동성 및 상승 비율 계산
- 종합적인 추세 판단 (상승/하락/횡보)

In [4]:
import numpy as np
from datetime import datetime

class ProgramTradingTrendAnalyzer:
    """프로그램매매 추세 분석 클래스"""
    
    def __init__(self, KiwoomAPI):
        self.KiwoomAPI = KiwoomAPI
    
    def parse_amount(self, val):
        """금액 문자열을 정수로 변환 (음수 처리 포함)"""
        if isinstance(val, str):
            # -- 기호는 음수를 의미
            is_negative = val.startswith('--')
            # 모든 기호 제거
            val_clean = val.replace("+", "").replace(",", "").replace("-", "")
            try:
                amt = int(val_clean)
                return -amt if is_negative else amt
            except:
                return 0
        return 0
    
    def fetch_all_day_data(self, stock_code, date=None):
        """장 시작부터 현재까지 모든 데이터 수집"""
        if date is None:
            date = datetime.now().strftime("%Y%m%d")
        
        print(f"전체 데이터 조회 중... [{stock_code}] {date}")
        result = self.KiwoomAPI.get_hourly_program_trading_paginated(stock_code, date, "1")
        
        if 'output' in result and result['output']:
            # 시간순 정렬
            data = sorted(result['output'], key=lambda x: x.get('tm', '000000'))
            # 장 시작(09:00) 이후 데이터만 필터링
            filtered_data = [d for d in data if d.get('tm', '000000') >= '090000']
            print(f"  → {len(filtered_data)}개 시점 데이터 수집 완료")
            return filtered_data
        return []
    
    def analyze_trend_smart(self, data):
        """스마트한 추세 분석"""
        if not data or len(data) < 2:
            return {"trend": "데이터 부족", "details": {}}
        
        # 순매수금액 추출
        net_amounts = [self.parse_amount(d.get('prm_netprps_amt', '0')) for d in data]
        net_quantities = [self.parse_amount(d.get('prm_netprps_qty', '0')) for d in data]
        
        # 1. 전반부/후반부 평균 비교
        mid_point = len(net_amounts) // 2
        first_half_avg = np.mean(net_amounts[:mid_point]) if mid_point > 0 else 0
        second_half_avg = np.mean(net_amounts[mid_point:]) if mid_point < len(net_amounts) else 0
        
        # 2. 선형 회귀를 통한 기울기 계산
        x = np.arange(len(net_amounts))
        coefficients = np.polyfit(x, net_amounts, 1)
        slope = coefficients[0]
        
        # 3. 변동성 계산
        volatility = np.std(net_amounts) if len(net_amounts) > 1 else 0
        avg_amount = np.mean(net_amounts)
        cv = (volatility / avg_amount * 100) if avg_amount != 0 else 0
        
        # 4. 구간별 증감 패턴
        changes = [net_amounts[i+1] - net_amounts[i] for i in range(len(net_amounts)-1)]
        positive_changes = sum(1 for c in changes if c > 0)
        positive_ratio = positive_changes / len(changes) if changes else 0
        
        # 5. 추세 판단 (종합)
        if abs(cv) < 5:  # 변동성이 매우 낮음
            trend = "횡보"
            confidence = "높음"
        elif slope > 0 and second_half_avg > first_half_avg * 1.05:
            trend = "강한 상승"
            confidence = "높음" if positive_ratio > 0.6 else "중간"
        elif slope > 0 and second_half_avg > first_half_avg:
            trend = "상승"
            confidence = "중간"
        elif slope < 0 and second_half_avg < first_half_avg * 0.95:
            trend = "강한 하락"
            confidence = "높음" if positive_ratio < 0.4 else "중간"
        elif slope < 0 and second_half_avg < first_half_avg:
            trend = "하락"
            confidence = "중간"
        else:
            trend = "횡보"
            confidence = "낮음"
        
        return {
            "trend": trend,
            "confidence": confidence,
            "details": {
                "first_half_avg": first_half_avg,
                "second_half_avg": second_half_avg,
                "change_ratio": ((second_half_avg - first_half_avg) / first_half_avg * 100) if first_half_avg != 0 else 0,
                "slope": slope,
                "volatility_cv": cv,
                "positive_change_ratio": positive_ratio * 100,
                "total_change": net_amounts[-1] - net_amounts[0],
                "start_value": net_amounts[0],
                "end_value": net_amounts[-1],
                "max_value": max(net_amounts),
                "min_value": min(net_amounts)
            },
            "quantities": {
                "start": net_quantities[0] if net_quantities else 0,
                "end": net_quantities[-1] if net_quantities else 0,
                "change": (net_quantities[-1] - net_quantities[0]) if net_quantities and len(net_quantities) > 1 else 0
            },
            "time_range": {
                "start": data[0].get('tm', 'N/A') if data else 'N/A',
                "end": data[-1].get('tm', 'N/A') if data else 'N/A'
            }
        }

# 분석기 초기화
analyzer = ProgramTradingTrendAnalyzer(KiwoomAPI)
print("프로그램매매 추세 분석기 초기화 완료")

프로그램매매 추세 분석기 초기화 완료


### 삼성전자 분석

In [5]:
# 삼성전자 프로그램매매 추세 분석
stock_code = "005930"
stock_name = "삼성전자"

print(f"{'='*70}")
print(f"📊 {stock_name}({stock_code}) 프로그램매매 추세 분석")
print(f"{'='*70}")

# 데이터 수집
data = analyzer.fetch_all_day_data(stock_code)

if data:
    # 추세 분석
    analysis = analyzer.analyze_trend_smart(data)
    
    print(f"\n⏰ 시간 범위: {analysis['time_range']['start']} ~ {analysis['time_range']['end']}")
    
    print(f"\n💰 순매수금액 분석:")
    print(f"  • 시작: {analysis['details']['start_value']:,}백만원")
    print(f"  • 현재: {analysis['details']['end_value']:,}백만원")
    print(f"  • 변화: {analysis['details']['total_change']:+,}백만원")
    print(f"  • 최대: {analysis['details']['max_value']:,}백만원")
    print(f"  • 최소: {analysis['details']['min_value']:,}백만원")
    
    print(f"\n📊 구간 분석:")
    print(f"  • 전반부 평균: {analysis['details']['first_half_avg']:,.0f}백만원")
    print(f"  • 후반부 평균: {analysis['details']['second_half_avg']:,.0f}백만원")
    print(f"  • 변화율: {analysis['details']['change_ratio']:+.2f}%")
    
    print(f"\n📉 추세 지표:")
    print(f"  • 선형 기울기: {analysis['details']['slope']:+.2f}")
    print(f"  • 변동성(CV): {analysis['details']['volatility_cv']:.2f}%")
    print(f"  • 상승 비율: {analysis['details']['positive_change_ratio']:.1f}%")
    
    print(f"\n🎯 추세 판단: **{analysis['trend']}** (신뢰도: {analysis['confidence']})")
    
    print(f"\n📦 순매수수량:")
    print(f"  • 시작: {analysis['quantities']['start']:,}주")
    print(f"  • 현재: {analysis['quantities']['end']:,}주")
    print(f"  • 변화: {analysis['quantities']['change']:+,}주")
else:
    print("데이터를 수집할 수 없습니다.")

📊 삼성전자(005930) 프로그램매매 추세 분석
전체 데이터 조회 중... [005930] 20251029
  → 3742개 시점 데이터 수집 완료

⏰ 시간 범위: 090003 ~ 153023

💰 순매수금액 분석:
  • 시작: 0백만원
  • 현재: -150,330백만원
  • 변화: -150,330백만원
  • 최대: 0백만원
  • 최소: -174,233백만원

📊 구간 분석:
  • 전반부 평균: -102,420백만원
  • 후반부 평균: -136,527백만원
  • 변화율: +33.30%

📉 추세 지표:
  • 선형 기울기: -20.21
  • 변동성(CV): -21.91%
  • 상승 비율: 35.0%

🎯 추세 판단: **강한 하락** (신뢰도: 높음)

📦 순매수수량:
  • 시작: 0주
  • 현재: -1,508,930주
  • 변화: -1,508,930주


### SK하이닉스 분석

In [16]:
# SK하이닉스 프로그램매매 추세 분석
stock_code = "000660"
stock_name = "SK하이닉스"

print(f"{'='*70}")
print(f"📊 {stock_name}({stock_code}) 프로그램매매 추세 분석")
print(f"{'='*70}")

# 데이터 수집
data = analyzer.fetch_all_day_data(stock_code)

if data:
    # 추세 분석
    analysis = analyzer.analyze_trend_smart(data)
    
    print(f"\n⏰ 시간 범위: {analysis['time_range']['start']} ~ {analysis['time_range']['end']}")
    
    print(f"\n💰 순매수금액 분석:")
    print(f"  • 시작: {analysis['details']['start_value']:,}백만원")
    print(f"  • 현재: {analysis['details']['end_value']:,}백만원")
    print(f"  • 변화: {analysis['details']['total_change']:+,}백만원")
    
    # 음수 처리를 위한 표시
    if analysis['details']['end_value'] < 0:
        print(f"  ⚠️ 현재 순매도 상태입니다!")
    
    print(f"  • 최대: {analysis['details']['max_value']:,}백만원")
    print(f"  • 최소: {analysis['details']['min_value']:,}백만원")
    
    print(f"\n📊 구간 분석:")
    print(f"  • 전반부 평균: {analysis['details']['first_half_avg']:,.0f}백만원")
    print(f"  • 후반부 평균: {analysis['details']['second_half_avg']:,.0f}백만원")
    print(f"  • 변화율: {analysis['details']['change_ratio']:+.2f}%")
    
    print(f"\n📉 추세 지표:")
    print(f"  • 선형 기울기: {analysis['details']['slope']:+.2f}")
    print(f"  • 변동성(CV): {abs(analysis['details']['volatility_cv']):.2f}%")
    print(f"  • 상승 비율: {analysis['details']['positive_change_ratio']:.1f}%")
    
    print(f"\n🎯 추세 판단: **{analysis['trend']}** (신뢰도: {analysis['confidence']})")
    
    print(f"\n📦 순매수수량:")
    print(f"  • 시작: {analysis['quantities']['start']:,}주")
    print(f"  • 현재: {analysis['quantities']['end']:,}주")
    print(f"  • 변화: {analysis['quantities']['change']:+,}주")
else:
    print("데이터를 수집할 수 없습니다.")

📊 SK하이닉스(000660) 프로그램매매 추세 분석
전체 데이터 조회 중... [000660] 20250922
  → 4319개 시점 데이터 수집 완료

⏰ 시간 범위: 090003 ~ 155239

💰 순매수금액 분석:
  • 시작: 0백만원
  • 현재: -68,034백만원
  • 변화: -68,034백만원
  ⚠️ 현재 순매도 상태입니다!
  • 최대: 6,195백만원
  • 최소: -124,284백만원

📊 구간 분석:
  • 전반부 평균: -45,355백만원
  • 후반부 평균: -103,229백만원
  • 변화율: +127.60%

📉 추세 지표:
  • 선형 기울기: -24.70
  • 변동성(CV): 42.91%
  • 상승 비율: 38.4%

🎯 추세 판단: **강한 하락** (신뢰도: 높음)

📦 순매수수량:
  • 시작: 0주
  • 현재: -191,347주
  • 변화: -191,347주


### 여러 종목 동시 비교

In [17]:
# 여러 종목 한번에 분석
stocks = [
    ("005930", "삼성전자"),
    ("000660", "SK하이닉스"),
    ("005380", "현대차"),
    ("051910", "LG화학")
]

results = []

for stock_code, stock_name in stocks:
    print(f"\n분석 중: {stock_name}({stock_code})")
    
    # 데이터 수집
    data = analyzer.fetch_all_day_data(stock_code)
    
    if data:
        # 추세 분석
        analysis = analyzer.analyze_trend_smart(data)
        
        results.append({
            "종목명": stock_name,
            "종목코드": stock_code,
            "추세": analysis['trend'],
            "신뢰도": analysis['confidence'],
            "순매수금액": analysis['details']['end_value'],
            "변화": analysis['details']['total_change'],
            "변화율": analysis['details']['change_ratio'],
            "상승비율": analysis['details']['positive_change_ratio']
        })

# 결과 요약 테이블
print(f"\n{'='*100}")
print(f"📊 종합 분석 결과")
print(f"{'='*100}")
print(f"{'종목명':^10} {'종목코드':^8} {'추세':^12} {'신뢰도':^6} {'순매수금액':>12} {'변화':>12} {'변화율':>8} {'상승비율':>8}")
print(f"{'-'*100}")

for r in results:
    print(f"{r['종목명']:10} {r['종목코드']:8} {r['추세']:12} {r['신뢰도']:6} "
          f"{r['순매수금액']:12,} {r['변화']:+12,} {r['변화율']:+7.1f}% {r['상승비율']:7.1f}%")

# 추세별 그룹핑
print(f"\n📈 추세별 분류:")
for trend_type in ["강한 상승", "상승", "횡보", "하락", "강한 하락"]:
    stocks_in_trend = [r['종목명'] for r in results if r['추세'] == trend_type]
    if stocks_in_trend:
        print(f"  • {trend_type}: {', '.join(stocks_in_trend)}")


분석 중: 삼성전자(005930)
전체 데이터 조회 중... [005930] 20250922
  → 4330개 시점 데이터 수집 완료

분석 중: SK하이닉스(000660)
전체 데이터 조회 중... [000660] 20250922


요청 처리 실패
Traceback (most recent call last):
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/pykiwoom_rest/base_api.py", line 312, in _make_request
    response.raise_for_status()
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: null for url: https://api.kiwoom.com/api/dostk/mrkcond
연속조회 요청 실패
Traceback (most recent call last):
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/pykiwoom_rest/kiwoom_base.py", line 467, in make_tr_request_continuous
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/pykiwoom_rest/base_api.py", line 312, in _make_request
    response.raise_for_status()
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=s

  → 4319개 시점 데이터 수집 완료

분석 중: 현대차(005380)
전체 데이터 조회 중... [005380] 20250922


요청 처리 실패
Traceback (most recent call last):
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/pykiwoom_rest/base_api.py", line 312, in _make_request
    response.raise_for_status()
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: null for url: https://api.kiwoom.com/api/dostk/mrkcond
연속조회 요청 실패
Traceback (most recent call last):
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/pykiwoom_rest/kiwoom_base.py", line 467, in make_tr_request_continuous
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/pykiwoom_rest/base_api.py", line 312, in _make_request
    response.raise_for_status()
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=s

  → 4136개 시점 데이터 수집 완료

분석 중: LG화학(051910)
전체 데이터 조회 중... [051910] 20250922


요청 처리 실패
Traceback (most recent call last):
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/pykiwoom_rest/base_api.py", line 312, in _make_request
    response.raise_for_status()
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: null for url: https://api.kiwoom.com/api/dostk/mrkcond
연속조회 요청 실패
Traceback (most recent call last):
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/pykiwoom_rest/kiwoom_base.py", line 467, in make_tr_request_continuous
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/pykiwoom_rest/base_api.py", line 312, in _make_request
    response.raise_for_status()
  File "/home/unohee/RTX_ENV/lib/python3.12/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=s

  → 3483개 시점 데이터 수집 완료

📊 종합 분석 결과
   종목명       종목코드        추세       신뢰도          순매수금액           변화      변화율     상승비율
----------------------------------------------------------------------------------------------------
삼성전자       005930   강한 상승        높음          658,406     +658,406   +58.6%    67.8%
SK하이닉스     000660   강한 하락        높음          -68,034      -68,034  +127.6%    38.4%
현대차        005380   강한 상승        중간           54,970      +54,970  +105.9%    53.0%
LG화학       051910   강한 하락        높음            3,131       +3,131    -7.9%    26.9%

📈 추세별 분류:
  • 강한 상승: 삼성전자, 현대차
  • 강한 하락: SK하이닉스, LG화학
